# 01 · Inicialización de embeddings para SLICES tipado

Este notebook amplía `seyonec/ChemBERTa-zinc-base-v1` de 767 a 899 embeddings e inicializa las 132 filas nuevas según su semántica. No entrena el modelo. Su único objetivo es producir un checkpoint inicial reproducible y auditable antes de congelar el encoder y comenzar SLICES-TAPT.

## Diseño

| Familia | Estrategia | Intención |
|---|---|---|
| `<EL_X>` transferible | Copiar el embedding del símbolo químico completo | Conservar conocimiento SMILES fiable |
| `<EL_X>` no transferible | Centro neutral + offsets ortogonales | Evitar interpretar fragmentos BPE como química |
| `<N_i>` | Centro de dígitos + offsets ortogonales | Identificadores distinguibles sin orden artificial |
| `<T_xyz>` | Centro común + combinación de tres ejes ortogonales | Introducir la geometría algebraica de traslación |
| `[SLICES]` | Centro de tokens especiales + perturbación pequeña | Marcar el cambio de modalidad |

Un elemento se considera transferible sólo cuando su símbolo completo es exactamente un token del vocabulario original. No se promedian fragmentos: `Fe → F e` no implica que hierro sea la media de flúor y una pieza BPE.

## Calibración radial

El plan define escalas relativas a la dispersión real del checkpoint:

$$r=\operatorname{median}_i\lVert W_i-\mu\rVert,\qquad \alpha_E=\alpha_N=0.05r,\quad \alpha_T=0.10r.$$

En este checkpoint, promediar embeddings cancela gran parte de su dirección: la media global y los centroides de dígitos/especiales tienen normas menores que una fila típica. Para no colocar tokens nuevos cerca del origen, cada centro recibe una **componente radial común, neutral y ortogonal**. Esta componente sólo identifica la familia y lleva su norma al manifold existente; las relaciones internas siguen siendo exactamente las del plan.

In [1]:
from hashlib import sha256
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForMaskedLM, AutoTokenizer

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'slices_tipado' / 'typed_vocabulary.csv').exists():
            return candidate
    raise FileNotFoundError('No se encontró el vocabulario SLICES tipado.')

PROJECT_ROOT = find_project_root(Path.cwd())
TYPED_ROOT = PROJECT_ROOT / 'data' / 'slices_tipado'
TYPED_TOKENIZER_DIR = TYPED_ROOT / 'chemberta_slices_tokenizer_v1'
VOCABULARY_PATH = TYPED_ROOT / 'typed_vocabulary.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'transferability' / 'initialize' / 'artifacts'
MODEL_DIR = OUTPUT_ROOT / 'chemberta_slices_initialized'
TABLE_PATH = OUTPUT_ROOT / 'initialization_table.csv'
GEOMETRY_PATH = OUTPUT_ROOT / 'initialization_geometry.npz'
MANIFEST_PATH = OUTPUT_ROOT / 'initialization_manifest.json'

MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'
MODEL_REVISION = '761d6a18cf99db371e0b43baf3e2d21b3e865a20'
CACHE_DIR = PROJECT_ROOT / '.cache' / 'huggingface'
RANDOM_SEED = 42
ALPHA_ELEMENT_RATIO = 0.05
ALPHA_NODE_RATIO = 0.05
ALPHA_TRANSLATION_RATIO = 0.10
ALPHA_SPECIAL_RATIO = 0.02

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
torch.manual_seed(RANDOM_SEED)
print(f'Salida: {OUTPUT_ROOT.relative_to(PROJECT_ROOT)}')

Salida: transferability/initialize/artifacts


## 1. Cargar vocabularios y checkpoint base

Se comprueba que el tokenizer tipado tenga exactamente los IDs descritos por `typed_vocabulary.csv` y que el checkpoint real corresponda a la arquitectura esperada.

In [2]:
vocabulary = pd.read_csv(VOCABULARY_PATH).sort_values('token_id').reset_index(drop=True)
base_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, cache_dir=CACHE_DIR
)
typed_tokenizer = AutoTokenizer.from_pretrained(TYPED_TOKENIZER_DIR)
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, cache_dir=CACHE_DIR
)
model.eval()

assert len(base_tokenizer) == 767
assert len(typed_tokenizer) == 899
assert len(vocabulary) == 132
assert vocabulary['token_id'].astype(int).tolist() == [
    typed_tokenizer.convert_tokens_to_ids(token) for token in vocabulary['token_tipado']
]

input_embeddings = model.get_input_embeddings()
base_weight = input_embeddings.weight.detach().cpu().float().clone()
OLD_VOCAB_SIZE, HIDDEN_SIZE = base_weight.shape
assert OLD_VOCAB_SIZE == len(base_tokenizer) == 767
assert HIDDEN_SIZE == model.config.hidden_size == 768
assert model.config.num_hidden_layers == 6
assert model.config.max_position_embeddings == 514

architecture = pd.DataFrame({
    'propiedad': ['modelo', 'vocabulario original', 'vocabulario tipado', 'hidden size',
                  'capas Transformer', 'posiciones máximas', 'weight tying original'],
    'valor': [type(model).__name__, OLD_VOCAB_SIZE, len(typed_tokenizer), HIDDEN_SIZE,
              model.config.num_hidden_layers, model.config.max_position_embeddings,
              model.get_input_embeddings().weight.data_ptr() == model.get_output_embeddings().weight.data_ptr()],
})
display(architecture)

Loading weights:   0%|          | 0/108 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,propiedad,valor
0,modelo,RobertaForMaskedLM
1,vocabulario original,767
2,vocabulario tipado,899
3,hidden size,768
4,capas Transformer,6
5,posiciones máximas,514
6,weight tying original,True


## 2. Estadística del espacio original

Todas las escalas se calculan desde las 767 filas preentrenadas. Ningún número depende de una distribución genérica de inicialización.

In [3]:
global_mean = base_weight.mean(dim=0)
base_radius = torch.linalg.vector_norm(base_weight - global_mean, dim=1)
base_norm = torch.linalg.vector_norm(base_weight, dim=1)
median_radius = base_radius.median()

digit_tokens = [str(value) for value in range(10)]
digit_ids = torch.tensor([base_tokenizer.convert_tokens_to_ids(token) for token in digit_tokens])
assert all(base_tokenizer.tokenize(token) == [token] for token in digit_tokens)
digit_mean = base_weight[digit_ids].mean(dim=0)

special_tokens = [base_tokenizer.bos_token, base_tokenizer.eos_token, base_tokenizer.mask_token]
special_ids = torch.tensor([base_tokenizer.convert_tokens_to_ids(token) for token in special_tokens])
special_mean = base_weight[special_ids].mean(dim=0)

alpha_element = ALPHA_ELEMENT_RATIO * median_radius
alpha_node = ALPHA_NODE_RATIO * median_radius
alpha_translation = ALPHA_TRANSLATION_RATIO * median_radius
alpha_special = ALPHA_SPECIAL_RATIO * median_radius

embedding_statistics = pd.DataFrame({
    'métrica': ['norma media global', 'norma centro dígitos', 'norma centro especiales',
                'radio mediano r', 'norma fila q01', 'norma fila mediana', 'norma fila q99',
                'alpha elemento', 'alpha nodo', 'alpha traslación', 'alpha marcador'],
    'valor': [global_mean.norm(), digit_mean.norm(), special_mean.norm(), median_radius,
              torch.quantile(base_norm, 0.01), base_norm.median(), torch.quantile(base_norm, 0.99),
              alpha_element, alpha_node, alpha_translation, alpha_special],
}).astype({'valor': float})
display(embedding_statistics)

,métrica,valor
0,norma media global,0.020157
1,norma centro dígitos,0.166052
2,norma centro especiales,0.318313
3,radio mediano r,0.553286
4,norma fila q01,0.521993
5,norma fila mediana,0.553607
6,norma fila q99,0.583841
7,alpha elemento,0.027664
8,alpha nodo,0.027664
9,alpha traslación,0.055329


## 3. Elementos transferibles y fallback neutral

El criterio de transferencia es deliberadamente estricto: `X` debe pertenecer al vocabulario original y `tokenize(X)` debe devolver exactamente `[X]`. Los demás elementos reciben un centro neutral calibrado y direcciones ortogonales pequeñas. No se usan propiedades periódicas externas en esta primera versión.

In [4]:
element_rows = vocabulary.query("familia == 'elemento'").copy()

def transferable_source(symbol: str) -> tuple[bool, int | None]:
    pieces = base_tokenizer.tokenize(symbol)
    token_id = base_tokenizer.convert_tokens_to_ids(symbol)
    reliable = (
        pieces == [symbol]
        and symbol in base_tokenizer.get_vocab()
        and token_id != base_tokenizer.unk_token_id
    )
    return reliable, int(token_id) if reliable else None

transferability = []
for row in element_rows.itertuples(index=False):
    reliable, source_id = transferable_source(row.token_original)
    transferability.append({
        'elemento': row.token_original, 'token_tipado': row.token_tipado,
        'transferible': reliable, 'token_fuente': row.token_original if reliable else '',
        'id_fuente': source_id,
        'tokenización_original': ' '.join(base_tokenizer.tokenize(row.token_original)),
    })
transferability_table = pd.DataFrame(transferability)
display(transferability_table)
display(transferability_table.groupby('transferible').size().rename('elementos').to_frame())
TRANSFERABLE_ELEMENTS = transferability_table.query('transferible')['elemento'].tolist()
FALLBACK_ELEMENTS = transferability_table.query('not transferible')['elemento'].tolist()
assert len(TRANSFERABLE_ELEMENTS) == 17
assert len(FALLBACK_ELEMENTS) == 67

,elemento,token_tipado,transferible,token_fuente,id_fuente,tokenización_original
0,H,<EL_H>,True,H,44.0,H
1,Li,<EL_Li>,False,,NaN,L i
2,Be,<EL_Be>,False,,NaN,B e
3,B,<EL_B>,True,B,38.0,B
4,C,<EL_C>,True,C,39.0,C
...,...,...,...,...,...,...
79,Ac,<EL_Ac>,False,,NaN,A c
80,Th,<EL_Th>,False,,NaN,T h
81,U,<EL_U>,True,U,57.0,U
82,Np,<EL_Np>,False,,NaN,N p


,elementos
transferible,
False,67
True,17


## 4. Construir geometría ortogonal y centros calibrados

Se genera una sola base ortonormal determinista. Sus direcciones son ortogonales entre sí y a los centroides originales. La calibración radial añade a cada familia una dirección común cuya magnitud hace que sus filas tengan aproximadamente la norma mediana del checkpoint.

Para traslaciones se calibra contra el número esperado de dos componentes no nulas entre las 27 combinaciones. Esto preserva exactamente composición y oposición alrededor de `<T_ooo>`.

In [5]:
def orthonormal_directions(count: int, dimension: int, exclude: list[torch.Tensor], seed: int) -> torch.Tensor:
    generator = torch.Generator(device='cpu').manual_seed(seed)
    fixed = torch.stack(exclude, dim=1)
    fixed_q, _ = torch.linalg.qr(fixed, mode='reduced')
    random_matrix = torch.randn(dimension, count, generator=generator)
    random_matrix = random_matrix - fixed_q @ (fixed_q.T @ random_matrix)
    directions, _ = torch.linalg.qr(random_matrix, mode='reduced')
    return directions.T.contiguous()

direction_count = 1 + len(FALLBACK_ELEMENTS) + 1 + 20 + 1 + 3 + 1 + 1
directions = orthonormal_directions(
    direction_count, HIDDEN_SIZE, [global_mean, digit_mean, special_mean], RANDOM_SEED
)
assert torch.allclose(directions @ directions.T, torch.eye(direction_count), atol=1e-5)

cursor = 0
element_center_direction = directions[cursor]; cursor += 1
element_offsets = directions[cursor:cursor + len(FALLBACK_ELEMENTS)]; cursor += len(FALLBACK_ELEMENTS)
node_center_direction = directions[cursor]; cursor += 1
node_offsets = directions[cursor:cursor + 20]; cursor += 20
translation_center_direction = directions[cursor]; cursor += 1
translation_axes = directions[cursor:cursor + 3]; cursor += 3
special_center_direction = directions[cursor]; cursor += 1
special_perturbation_direction = directions[cursor]; cursor += 1
assert cursor == direction_count

def calibrated_center(raw_center: torch.Tensor, center_direction: torch.Tensor,
                      expected_offset_sq: torch.Tensor | float) -> tuple[torch.Tensor, float]:
    available_sq = median_radius.square() - raw_center.norm().square() - torch.as_tensor(expected_offset_sq)
    if available_sq <= 0:
        raise ValueError('El centro original ya excede el radio objetivo.')
    beta = torch.sqrt(available_sq)
    return raw_center + beta * center_direction, float(beta)

element_fallback_center, beta_element = calibrated_center(
    global_mean, element_center_direction, alpha_element.square()
)
node_center, beta_node = calibrated_center(
    digit_mean, node_center_direction, alpha_node.square()
)
translation_center, beta_translation = calibrated_center(
    global_mean, translation_center_direction, 2 * alpha_translation.square()
)
slices_center, beta_special = calibrated_center(
    special_mean, special_center_direction, alpha_special.square()
)

center_table = pd.DataFrame({
    'familia': ['elemento fallback', 'nodo', 'traslación', '[SLICES]'],
    'norma centro crudo': [global_mean.norm(), digit_mean.norm(), global_mean.norm(), special_mean.norm()],
    'beta radial': [beta_element, beta_node, beta_translation, beta_special],
    'norma centro calibrado': [element_fallback_center.norm(), node_center.norm(),
                                 translation_center.norm(), slices_center.norm()],
}).astype({column: float for column in ['norma centro crudo', 'beta radial', 'norma centro calibrado']})
display(center_table)

,familia,norma centro crudo,beta radial,norma centro calibrado
0,elemento fallback,0.020157,0.552226,0.552594
1,nodo,0.166052,0.527055,0.552594
2,traslación,0.020157,0.547354,0.547725
3,[SLICES],0.318313,0.452415,0.553175


## 5. Redimensionar e inicializar las 132 filas

Las 767 filas originales deben permanecer bit a bit iguales. Después del resize se sobreescribe cada fila nueva según su familia y se vuelve a ejecutar `tie_weights()`.

In [6]:
model.resize_token_embeddings(len(typed_tokenizer), mean_resizing=False)
model.tie_weights()
weight = model.get_input_embeddings().weight
assert weight.shape == (899, HIDDEN_SIZE)
assert torch.equal(weight[:OLD_VOCAB_SIZE].detach().cpu().float(), base_weight)

fallback_index = {symbol: index for index, symbol in enumerate(FALLBACK_ELEMENTS)}
node_index = {str(value): value for value in range(20)}
translation_values = {'-': -1.0, 'o': 0.0, '+': 1.0}
initialization_records = []

with torch.no_grad():
    for row in vocabulary.itertuples(index=False):
        token_id = int(row.token_id)
        source_token = ''
        source_id = None

        if row.familia == 'marcador':
            vector = slices_center + alpha_special * special_perturbation_direction
            strategy = 'special_centroid_calibrated_plus_small_offset'
        elif row.familia == 'elemento':
            reliable, source_id = transferable_source(row.token_original)
            if reliable:
                vector = base_weight[source_id].clone()
                source_token = row.token_original
                strategy = 'exact_chemical_token_transfer'
            else:
                vector = element_fallback_center + alpha_element * element_offsets[fallback_index[row.token_original]]
                strategy = 'neutral_calibrated_center_plus_orthogonal_offset'
        elif row.familia == 'nodo':
            vector = node_center + alpha_node * node_offsets[node_index[row.token_original]]
            strategy = 'digit_centroid_calibrated_plus_orthogonal_offset'
        elif row.familia == 'traslación':
            coordinates = torch.tensor([translation_values[value] for value in row.token_original])
            vector = translation_center + alpha_translation * (coordinates @ translation_axes)
            strategy = 'calibrated_center_plus_structured_3d_basis'
        else:
            raise ValueError(f'Familia desconocida: {row.familia}')

        weight[token_id].copy_(vector.to(device=weight.device, dtype=weight.dtype))
        initialization_records.append({
            'token': row.token_tipado, 'token_id': token_id, 'familia': row.familia,
            'token_original': row.token_original, 'estrategia': strategy,
            'token_fuente': source_token, 'id_fuente': source_id,
        })

model.tie_weights()
initialized_weight = model.get_input_embeddings().weight.detach().cpu().float()
assert torch.equal(initialized_weight[:OLD_VOCAB_SIZE], base_weight)
assert model.get_input_embeddings().weight.data_ptr() == model.get_output_embeddings().weight.data_ptr()

initialization_table = pd.DataFrame(initialization_records)
new_vectors = initialized_weight[initialization_table['token_id'].to_numpy()]
initialization_table['norma'] = torch.linalg.vector_norm(new_vectors, dim=1).numpy()
initialization_table['distancia_media_vocabulario'] = torch.linalg.vector_norm(
    new_vectors - global_mean, dim=1
).numpy()
display(initialization_table.groupby(['familia', 'estrategia'], sort=False).agg(
    tokens=('token', 'size'), norma_mín=('norma', 'min'), norma_media=('norma', 'mean'),
    norma_máx=('norma', 'max')
))

tokens  \
familia    estrategia                                                 
marcador   special_centroid_calibrated_plus_small_offset          1   
elemento   exact_chemical_token_transfer                         17   
           neutral_calibrated_center_plus_orthogonal_offset      67   
nodo       digit_centroid_calibrated_plus_orthogonal_offset      20   
traslación calibrated_center_plus_structured_3d_basis            27   

                                                             norma_mín  \
familia    estrategia                                                    
marcador   special_centroid_calibrated_plus_small_offset      0.553286   
elemento   exact_chemical_token_transfer                      0.532662   
           neutral_calibrated_center_plus_orthogonal_offset   0.553286   
nodo       digit_centroid_calibrated_plus_orthogonal_offset   0.553286   
traslación calibrated_center_plus_structured_3d_basis         0.547725   

                                                             norma_media  \
familia    estrategia                                                      
marcador   special_centroid_calibrated_plus_small_offset        0.553286   
elemento   exact_chemical_token_transfer                        0.555428   
           neutral_calibrated_center_plus_orthogonal_offset     0.553286   
nodo       digit_centroid_calibrated_plus_orthogonal_offset     0.553286   
traslación calibrated_center_plus_structured_3d_basis           0.553281   

                                                             norma_máx  
familia    estrategia                                                   
marcador   special_centroid_calibrated_plus_small_offset      0.553286  
elemento   exact_chemical_token_transfer                      0.579668  
           neutral_calibrated_center_plus_orthogonal_offset   0.553286  
nodo       digit_centroid_calibrated_plus_orthogonal_offset   0.553286  
traslación calibrated_center_plus_structured_3d_basis         0.556045

## 6. Comprobaciones mínimas antes de guardar

El audit independiente repetirá y ampliará estas pruebas. Aquí se bloquea inmediatamente un checkpoint con filas base alteradas, pesos desligados o relaciones geométricas incorrectas.

In [7]:
token_to_id = dict(zip(initialization_table['token'], initialization_table['token_id']))

def embedding(token: str) -> torch.Tensor:
    return initialized_weight[token_to_id[token]]

translation_zero = embedding('<T_ooo>')
opposite_error = torch.max(torch.abs(
    (embedding('<T_+oo>') - translation_zero)
    + (embedding('<T_-oo>') - translation_zero)
)).item()
composition_error = torch.max(torch.abs(
    (embedding('<T_++o>') - translation_zero)
    - (embedding('<T_+oo>') - translation_zero)
    - (embedding('<T_o+o>') - translation_zero)
)).item()

node_vectors = torch.stack([embedding(f'<N_{value}>') for value in range(20)])
node_centered = node_vectors - node_center
node_gram = node_centered @ node_centered.T / alpha_node.square()
node_orthogonality_error = torch.max(torch.abs(node_gram - torch.eye(20))).item()

quick_checks = pd.DataFrame({
    'comprobación': ['filas originales intactas', 'weight tying', 'oposición T ±x',
                    'composición T (+x + +y)', 'offsets de nodos ortonormales'],
    'valor': [True, True, opposite_error, composition_error, node_orthogonality_error],
    'pasa': [True, True, opposite_error < 1e-6, composition_error < 1e-6, node_orthogonality_error < 1e-5],
})
display(quick_checks)
assert quick_checks['pasa'].all()

,comprobación,valor,pasa
0,filas originales intactas,True,True
1,weight tying,True,True
2,oposición T ±x,0.0,True
3,composición T (+x + +y),0.0,True
4,offsets de nodos ortonormales,0.0,True


## 7. Guardar checkpoint, geometría y manifiesto

La geometría se guarda por separado para que el audit pueda verificar las ecuaciones completas sin volver a ejecutar la inicialización. El checkpoint incluye modelo y tokenizer, pero aún no incluye un optimizador ni entrenamiento.

In [8]:
model.config.slices_initialization = {
    'schema': 'SLICES-embedding-init-v1', 'seed': RANDOM_SEED,
    'base_model': MODEL_NAME, 'base_revision': MODEL_REVISION,
    'old_vocab_size': OLD_VOCAB_SIZE, 'new_vocab_size': len(typed_tokenizer),
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MODEL_DIR, safe_serialization=True)
typed_tokenizer.save_pretrained(MODEL_DIR)
initialization_table.to_csv(TABLE_PATH, index=False)

np.savez_compressed(
    GEOMETRY_PATH,
    global_mean=global_mean.numpy(), digit_mean=digit_mean.numpy(), special_mean=special_mean.numpy(),
    element_fallback_center=element_fallback_center.numpy(), element_offsets=element_offsets.numpy(),
    node_center=node_center.numpy(), node_offsets=node_offsets.numpy(),
    translation_center=translation_center.numpy(), translation_axes=translation_axes.numpy(),
    slices_center=slices_center.numpy(), special_perturbation_direction=special_perturbation_direction.numpy(),
    median_radius=np.array(float(median_radius)), alpha_element=np.array(float(alpha_element)),
    alpha_node=np.array(float(alpha_node)), alpha_translation=np.array(float(alpha_translation)),
    alpha_special=np.array(float(alpha_special)),
)

checkpoint_files = sorted(path for path in MODEL_DIR.iterdir() if path.is_file())
manifest = {
    'schema': 'SLICES-embedding-init-v1',
    'base_model': {'name': MODEL_NAME, 'revision': MODEL_REVISION},
    'architecture': {
        'old_vocab_size': OLD_VOCAB_SIZE, 'new_vocab_size': len(typed_tokenizer),
        'hidden_size': HIDDEN_SIZE, 'num_hidden_layers': model.config.num_hidden_layers,
        'max_position_embeddings': model.config.max_position_embeddings,
    },
    'seed': RANDOM_SEED,
    'scales': {
        'median_radius': float(median_radius), 'alpha_element': float(alpha_element),
        'alpha_node': float(alpha_node), 'alpha_translation': float(alpha_translation),
        'alpha_special': float(alpha_special), 'beta_element': beta_element,
        'beta_node': beta_node, 'beta_translation': beta_translation, 'beta_special': beta_special,
    },
    'families': {
        'marker': 1, 'elements': len(element_rows),
        'transferable_elements': TRANSFERABLE_ELEMENTS, 'fallback_elements': FALLBACK_ELEMENTS,
        'nodes': 20, 'translations': 27,
    },
    'checks': {
        'base_rows_unchanged': True, 'weight_tied': True,
        'translation_opposite_max_error': opposite_error,
        'translation_composition_max_error': composition_error,
        'node_orthogonality_max_error': node_orthogonality_error,
    },
    'files': {
        path.name: {'bytes': path.stat().st_size, 'sha256': sha256(path.read_bytes()).hexdigest()}
        for path in checkpoint_files
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

print(f'Modelo: {MODEL_DIR.relative_to(PROJECT_ROOT)}')
print(f'Tabla: {TABLE_PATH.relative_to(PROJECT_ROOT)}')
print(f'Geometría: {GEOMETRY_PATH.relative_to(PROJECT_ROOT)}')
print(f'Manifiesto: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo: transferability/initialize/artifacts/chemberta_slices_initialized
Tabla: transferability/initialize/artifacts/initialization_table.csv
Geometría: transferability/initialize/artifacts/initialization_geometry.npz
Manifiesto: transferability/initialize/artifacts/initialization_manifest.json


## 8. Nota crítica para la primera fase de entrenamiento

`requires_grad=False` no puede congelar filas individuales porque toda la matriz es un solo `Parameter`. Para entrenar únicamente IDs 767–898 se necesita un hook que ponga a cero las filas antiguas del gradiente y un grupo de optimizador con `weight_decay=0` para la matriz. El hook debe reinstalarse al cargar el modelo porque no se serializa.

```python
embedding_weight = model.get_input_embeddings().weight
old_vocab_size = 767

def mask_old_embedding_gradients(gradient):
    gradient = gradient.clone()
    gradient[:old_vocab_size].zero_()
    return gradient

hook = embedding_weight.register_hook(mask_old_embedding_gradients)
# El parameter group de embedding_weight debe usar weight_decay=0.
```

El audit separado debe pasar antes de iniciar esa fase.